# Fine-Tuning **SmolLM-135M** model for Generating Sports News:
 
 - Utilized **AG News Dataset** for a generative task using the **Sports** news available in it.
    

In [ ]:
# Importing Libraries:

import numpy as np
import pandas as pd
import torch

import datasets
from datasets import load_dataset

import transformers

# Preprocessing:
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling

# Training:
from transformers import TrainingArguments, Trainer

# Post Training Analysis:
from transformers import pipeline


## Loading **AG News** Dataset:


In [2]:
news_dataset = load_dataset("ag_news")
news_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
# Training Dataset and features:
news_train_dataset = news_dataset["train"]
print(news_train_dataset.features)


{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'], id=None)}


In [4]:
# Sports News:
news_train_dataset[1300]

{'text': "Phelps's chase of Spitz mark? It's history This was the event Michael Phelps didn't really need to compete in if his goal was to win eight golds. He probably would have had a better chance somewhere else.",
 'label': 1}

In [5]:
# Defining news id to label map:
news_id_to_label_map = { 0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech" }

In [6]:
# Filtering the sports news dataset using label_id:
sports_datasets = news_dataset.filter(lambda example: example["label"] == 1)
sports_datasets = sports_datasets.remove_columns("label")

## Preprocessing:

### Loading the tokenizer for SmolLM-135M:


In [7]:
# Loading tokenizer for SmolLM-135M: 
model_name = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [8]:
# We need to specify as SmolLM's tokenizer doesn't include the padding token:
tokenizer.pad_token = ( tokenizer.eos_token )  

In [9]:
# Define Tokenizer wrapper function:
def tokenizer_wrapper(batch):
    return tokenizer(batch["text"], truncation=True)


### Tokenizing the Sports News Dataset:

In [10]:
# Note: We require input_ids and attention_mask
# Since we can straight up work with token ids:

tokenized_sports_news_datasets = sports_datasets.map(
                                    tokenizer_wrapper,  
                                    batched = True, 
                                    remove_columns = ["text"],  
                                )


In [11]:
tokenized_sports_news_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1900
    })
})

In [12]:
# Showing example tokenization:
ip_index = 69
example_tokenized = tokenized_sports_news_datasets['train'][ip_index]
example_tokenized_input_ids = list(example_tokenized['input_ids'])
example_tokenized_attention_mask = list(example_tokenized['attention_mask'])

# Showing example tokenization:
print(f"tokenized_ids:\n{example_tokenized_input_ids}")
print(f"\n\nattention_mask:\n{example_tokenized_attention_mask}")



tokenized_ids:
[44745, 917, 8992, 18968, 370, 216, 35, 29, 33, 288, 48750, 31732, 534, 365, 3872, 25, 6594, 731, 41710, 8581, 12713, 3917, 582, 1658, 281, 2976, 7954, 616, 327, 650, 808, 4726, 281, 3920, 253, 3531, 284, 4573, 4653, 10463, 2994, 253, 1296, 29, 10521, 24190, 282, 260, 11554, 18968, 370, 351, 253, 216, 35, 29, 33, 9970, 10528, 30]


attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Training the Model for Fine-Tuning:


In [13]:
# Identifying device to train on GPU:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [14]:
# The parameter `mlm` ==> masked language modeling
# Since we are doing Causal Learning, we set:
# mlm = False

# Initialising the data collator:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [15]:
# Reviewing shape of the tokenized texts inputs:
samples = [tokenized_sports_news_datasets["train"][i] for i in range(5)]

for sample in samples:
    print(f"input_ids shape: {len(sample['input_ids'])}")

input_ids shape: 103
input_ids shape: 81
input_ids shape: 60
input_ids shape: 65
input_ids shape: 51


In [16]:
# Reviewing shape of the tokenized samples post using data collator:
data_collator_samples_output = data_collator(samples)
for key in data_collator_samples_output:
    print(f"{key} shape: {data_collator_samples_output[key].shape}")

input_ids shape: torch.Size([5, 103])
attention_mask shape: torch.Size([5, 103])
labels shape: torch.Size([5, 103])


### Loading the SmolLM-135M model:

In [17]:
# Loading the model (SmolLM-135M) for causal learning:
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

### Setting up Training Arguments and Initialising the Trainer:

In [18]:
# Setting the Training Arguments:
batch_size = 2

training_args = TrainingArguments(
    "sports-news-generator",
    push_to_hub = False,
    per_device_train_batch_size = batch_size,
    weight_decay = 0.1,
    lr_scheduler_type = "cosine",
    learning_rate = 5e-4,
    num_train_epochs = 2,
    eval_strategy = "steps",
    eval_steps = 200,
    logging_steps = 200,
)

In [19]:
# Shuffling dataset to pick 20000 examples to Train/Fine-Tune over:
shuffled_dataset = tokenized_sports_news_datasets["train"].shuffle(seed = 69)
training_subset_data = shuffled_dataset.select(range(20000))


In [20]:
# Initialize the Trainer:
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    args = training_args,
    data_collator = data_collator,
    train_dataset = training_subset_data,
    eval_dataset = tokenized_sports_news_datasets["test"],
)

/tmp/ipykernel_3620012/3803243364.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start Training/Fine-Tuning the model:

In [21]:
# Train the model:
trainer.train()

Step,Training Loss,Validation Loss
200,4.247100,4.227316
400,4.094400,4.170657
600,3.994500,4.128116
800,4.008400,4.051703
1000,3.985500,4.027221
1200,3.922800,3.983801
1400,3.928800,3.951313
1600,3.942000,3.918857
1800,3.846600,3.869408
2000,3.876000,3.847630


TrainOutput(global_step=20000, training_loss=2.889854670715332, metrics={'train_runtime': 4015.6466, 'train_samples_per_second': 9.961, 'train_steps_per_second': 4.981, 'total_flos': 1645061796472320.0, 'train_loss': 2.889854670715332, 'epoch': 2.0})

### Saving the Fine-Tuned model:

In [22]:
# Saving the Fine-Tuned model in './Sports_News_Generation_model' directory:
trainer.save_model("./Sports_News_Generation_model/sports_news_gen_model_bs2_ep3_FT_latest")


## Post Training/Fine-Tuning Analysis: 

In [25]:
# Initialising pipeline for inferences:
pipe = pipeline(
    "text-generation",
    model="./Sports_News_Generation_model/sports_news_gen_model_bs2_ep3_FT_latest", 
    device=device,
)



# Test generation:
print(
    pipe("1st Quarter", do_sample=True, temperature=0.1, max_new_tokens=30)[0][
        "generated_text"
    ]
)

Device set to use cuda


1st Quarterback, 2004, Wins MVP Battle (AP) AP - Tom Brady threw for 163 yards and two touchdowns


In [30]:
# Example generation:
input_prompt_example_1 = "Manchester City Champions League Final"
generated_example_1 = pipe(input_prompt_example_1, do_sample=True, temperature=0.1, max_new_tokens=30)[0][
        "generated_text"
    ]

input_prompt_example_2 = "Sacramento Kings"
generated_example_2 = pipe(input_prompt_example_2, do_sample=True, temperature=0.1, max_new_tokens=30)[0][
        "generated_text"
    ]


print(f"input_prompt_example_1:\n{input_prompt_example_1}\n\ngenerated_example_1:\n{generated_example_1} ")
print(f"input_prompt_example_2:\n{input_prompt_example_2}\n\ngenerated_example_2:\n{generated_example_2} ")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


input_prompt_example_1:
Manchester City Champions League Final

generated_example_1:
Manchester City Champions League Finale a Successive Game The final of the Champions League is a success for Manchester City. The club beat the French side 3-0 to move 
input_prompt_example_2:
Sacramento Kings

generated_example_2:
Sacramento Kings Team Report - November 16 (Sports Network) - The Sacramento Kings will try to get back on the winning track this evening when they meet the 


In [ ]:
import torch
torch.cuda.empty_cache()
